In [ ]:
# Import the usual libraries and variables.
%run standard_imports.py

# Install a pip package in the current Jupyter kernel.
import sys
!{sys.executable} -m pip install --user pypika

In [ ]:
import datetime
from dateutil.relativedelta import relativedelta
from pypika import MySQLQuery, Table, Order, Case, functions as fn

# Year to date
start_date = datetime.datetime.utcnow().replace(month=1, day=1, hour=0, minute=0, second=0, microsecond=0)
end_date = start_date + relativedelta(years=1)

# Instantiate tables
charge = Table("charge")
merchant_app_charge = Table("merchant_app_charge")
merchant_app = Table("merchant_app")
developer_app = Table("developer_app")
developer = Table("developer")

# Query gross revenue
gross_revenue = (fn.Sum(charge.amount)/100).as_("gross_revenue")
gross_developer_portion = (fn.Sum(fn.Coalesce(charge.developer_portion,
                                              charge.amount * 0.7))/100).as_("gross_developer_portion")
gross_clover_portion = (fn.Sum(charge.amount - fn.Coalesce(charge.developer_portion,
                                                           charge.amount * 0.7))/100).as_("gross_clover_portion")

q = MySQLQuery.from_(charge).join(
    merchant_app_charge
).on(
    merchant_app_charge.charge_id == charge.id 
).join(
    merchant_app
).on(
    merchant_app.id == merchant_app_charge.merchant_app_id
).join(
    developer_app
).on(
    developer_app.id == merchant_app.app_id
).join(
    developer
).on(
    developer.id == developer_app.developer_id
).select(
    developer.id,
    developer.uuid,
    developer.name,
    gross_revenue
).where(
    (developer.name != "Clover") &
    (developer.is_rev_share_flat_rate == 0)
).where(
    (charge.system_type == "INFOLEASE") &
    (charge.status.isin(["DISBURSED", "BILLED"])) &
    (charge.currency == "USD")  # How do we want to handle CAD? Add as-is or convert to USD at what rate?
).where(
    charge.created_time[start_date:end_date]
).groupby(developer.id) \
.orderby(gross_revenue, order=Order.desc)

gross_query = q.get_sql()
print(gross_query)

# Query refunds
# TODO: Functionalize this and the above, passing the charge status array & currency.
# TODO: Check between date -- slicing syntax implies end date is excluded, but is that true? Is that what I want?

total_refunds = (fn.Sum(charge.amount)/100).as_("total_refunds")  # Repetitive w/ gross_revenue.

q = MySQLQuery.from_(charge).join(
    merchant_app_charge
).on(
    merchant_app_charge.charge_id == charge.id 
).join(
    merchant_app
).on(
    merchant_app.id == merchant_app_charge.merchant_app_id
).join(
    developer_app
).on(
    developer_app.id == merchant_app.app_id
).join(
    developer
).on(
    developer.id == developer_app.developer_id
).select(
    developer.id,
#     developer.uuid,
#     developer.name,
    total_refunds
).where(
    (developer.name != "Clover") &
    (developer.is_rev_share_flat_rate == 0)
).where(
    (charge.system_type == "INFOLEASE") &
    (charge.status.isin(["REFUND_DOWNGRADE_PENDING", "REFUND_DOWNGRADE_IN_PROGRESS", "REFUNDED_DOWNGRADE"])) &
    (charge.currency == "USD")  # How do we want to handle CAD? Add as-is or convert to USD at what rate?
).where(
    charge.created_time[start_date:end_date]
).groupby(developer.id)

refund_query = q.get_sql()
print(refund_query)

db = Db("~/.clover/p801.cfg")
df = pd.read_sql(gross_query, con=db.conn)
print(df)
refunds = pd.read_sql(refund_query, con=db.conn)
print(refunds)
db.close()

In [ ]:
df = df.set_index("id").join(refunds.set_index("id"))
df.fillna(0, inplace=True)
df["net_revenue"] = df.apply(lambda row: row.gross_revenue - row.total_refunds, axis=1)

In [ ]:
import datetime
import calendar

def annualize_ytd(value):
    current = datetime.datetime.utcnow()
    day_of_year = current.timetuple().tm_yday
    total_days = 366 if calendar.isleap(current.year) else 365
    year_progress = float(day_of_year)/float(total_days)
    return value/year_progress

df["annualized_net_revenue"] = df["net_revenue"].map(lambda x: round(annualize_ytd(x)))
df

In [ ]:
DIAMOND_REV_THRESHOLD = 2400000
GOLD_REV_THRESHOLD    =  600000
SILVER_REV_THRESHOLD  =  250000
BRONZE_REV_THRESHOLD  =   50000

def categorize_by_rev(value):
    if value > DIAMOND_REV_THRESHOLD:
        return "Diamond"
    if value > GOLD_REV_THRESHOLD:
        return "Gold"
    if value > SILVER_REV_THRESHOLD:
        return "Silver"
    if value > BRONZE_REV_THRESHOLD:
        return "Bronze"
    else:
        return "Public"


df["support_tier_by_rev"] = df["annualized_net_revenue"].map(lambda x: categorize_by_rev(x))
df.sort_values(by="annualized_net_revenue", ascending=False, inplace=True)
df

In [ ]:
from pypika import Case

# Net merchant installs

# tables
merchant_app = Table("merchant_app")
merchant = Table("merchant")
merchant_gateway = Table("merchant_gateway")
payment_processor = Table("payment_processor")
developer_app = Table("developer_app")
developer = Table("developer")

net_installs = fn.Sum(
    Case()
    .when(merchant_app.deleted_time.isnull(), 1)
    .else_(0)
).as_("net_installs")

q = MySQLQuery.from_(merchant_app).join(
    merchant
).on(
    merchant.id == merchant_app.merchant_id
).join(
    merchant_gateway
).on(
    merchant_gateway.merchant_id == merchant_app.merchant_id
).join(
    payment_processor
).on(
    payment_processor.id == merchant_gateway.payment_processor_id
).join(
    developer_app
).on(
    developer_app.id == merchant_app.app_id
).join(
    developer
).on(
    developer.id == developer_app.developer_id  
).select(
    developer.id,
    net_installs
).where(
    (developer.name != "Clover") &
    (developer.is_rev_share_flat_rate == 0)
).where(
    (payment_processor.production == 1) &
    (merchant.infolease_suppress_billing != 1)    
).groupby(developer.id) \
.orderby(net_installs, order=Order.desc)

installs_query = q.get_sql()
print(installs_query)

db = Db("~/.clover/p801.cfg")
installs = pd.read_sql(installs_query, con=db.conn)
print(installs)
db.close()

In [ ]:
df = df.join(installs.set_index("id"))

In [ ]:
DIAMOND_INSTALLS_THRESHOLD = 50000
GOLD_INSTALLS_THRESHOLD    =  6000
SILVER_INSTALLS_THRESHOLD  =  1500
BRONZE_INSTALLS_THRESHOLD  =   300

def categorize_by_installs(value):
    if value > DIAMOND_INSTALLS_THRESHOLD:
        return "Diamond"
    if value > GOLD_INSTALLS_THRESHOLD:
        return "Gold"
    if value > SILVER_INSTALLS_THRESHOLD:
        return "Silver"
    if value > BRONZE_INSTALLS_THRESHOLD:
        return "Bronze"
    else:
        return "Public"
    
df["support_tier_by_installs"] = df["net_installs"].map(lambda x: categorize_by_installs(x))
df